<h1>Chapter 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This notebook is for Chapter 2 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'

In [3]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

# 1. Посмотреть, что происходит без <|assistant|>

Убери тег из промпта и сравни вывод:

In [12]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=36
)

# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing


Это наглядно покажет разницу между "чистым автодополнением" (модель может просто продолжить писать похожий текст задания, а не отвечать) и "режимом ассистента" — прямая иллюстрация того, зачем вообще нужны специальные токены и chat template.

# 2. Заглянуть внутрь: распределение вероятностей на каждом шаге

Самое ценное для понимания — увидеть, что модель не выбирает токен волшебным образом, а получает распределение вероятностей по всему словарю:

In [14]:
outputs = model(input_ids)
logits = outputs.logits[0, -1, :]  # логиты для *следующего* токена после промпта
probs = logits.softmax(dim=-1)

top_k = probs.topk(10)
for prob, idx in zip(top_k.values, top_k.indices):
    print(f"{tokenizer.decode(idx):>15} — {prob.item():.2%}")

            Sub — 85.94%
              D — 11.62%
             Hi — 0.35%
              
 — 0.35%
             ** — 0.27%
          Email — 0.27%
          Sarah — 0.17%
            ``` — 0.13%
              [ — 0.08%
          Hello — 0.06%


Это покажет топ-5 кандидатов на первый сгенерированный токен с их вероятностями — реальное "под капотом" того, что означает "жадный decoding": просто взять токен с максимальной вероятностью.

# 3. Сравнить стратегии декодирования

Ты уже видел do_sample=False (greedy). Теперь сравни с другими стратегиями на одном и том же промпте:

In [16]:
strategies = {
    "greedy": dict(do_sample=False),
    "sampling (temp=0.7)": dict(do_sample=True, temperature=0.7, top_p=0.9),
    "sampling (temp=1.5)": dict(do_sample=True, temperature=1.5),
    "beam search (n=3)": dict(num_beams=3, do_sample=False),
}

for name, kwargs in strategies.items():
    output = model.generate(input_ids=input_ids, max_new_tokens=36, **kwargs)
    print(f"--- {name} ---")
    print(tokenizer.decode(output[0][input_ids.shape[1]:]))  # только новые токены
    print()

--- greedy ---
Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing

--- sampling (temp=0.7) ---
Subject: Sincere Apologies for the Gardening Incident


Dear Sarah,


I hope this message finds you well. I am writing to

--- sampling (temp=1.5) ---
Subject: Heartfelt Apologies for Our Unfounded Gardens' Tragedy


Dear Sarah,


I hope this message finds you in

--- beam search (n=3) ---
Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing



Отличная база для главы про decoding strategies — наглядно видно, как temperature=1.5 начинает генерировать что-то странное и несвязное, а beam search даёт более "выверенный", но иногда шаблонный текст.

# 4. Влияние max_new_tokens на обрыв мысли

In [17]:
for n in [5, 10, 20, 50, 100]:
    output = model.generate(input_ids=input_ids, max_new_tokens=n, do_sample=False)
    print(f"--- max_new_tokens={n} ---")
    print(tokenizer.decode(output[0][input_ids.shape[1]:]))
    print()

--- max_new_tokens=5 ---
Subject: Sinc

--- max_new_tokens=10 ---
Subject: Sincere Apologies for the

--- max_new_tokens=20 ---
Subject: Sincere Apologies for the Gardening Mishap


Dear

--- max_new_tokens=50 ---
Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that

--- max_new_tokens=100 ---
Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have always admired the beauty and tranquility of your garden. It was with great regret that I decided to help you with the weeding process, hoping to contribute to the



Покажет, что при малом лимите текст обрывается на полуслове — важная иллюстрация того, что модель не "знает", что ответ должен уместиться в лимит, она просто генерирует до тех пор, пока её не остановят.

# 5. Токенизация: сравнить разные модели токенайзеров

Раз у тебя уже есть Phi-3 токенайзер, сравни с другим (например, GPT-2 или BERT), чтобы показать, что токенизация — не универсальна:

In [18]:
from transformers import AutoTokenizer

other_tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Gardening mishap"
print("Phi-3:", tokenizer.tokenize(text))
print("GPT-2:", other_tokenizer.tokenize(text))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Phi-3: ['▁Garden', 'ing', '▁m', 'ish', 'ap']
GPT-2: ['G', 'ard', 'ening', 'Ġmish', 'ap']


Разное количество токенов и разбиение на подслова у разных моделей — хорошая тема для главы про токенизацию (если она у тебя ещё не написана или планируется).

# 6. Посчитать время генерации на токен

Практический аспект, полезный для главы про production/производительность:

In [19]:
import time

start = time.time()
output = model.generate(input_ids=input_ids, max_new_tokens=100, do_sample=False)
elapsed = time.time() - start

new_tokens = output.shape[1] - input_ids.shape[1]
print(f"Сгенерировано {new_tokens} токенов за {elapsed:.2f}с ({new_tokens/elapsed:.1f} токенов/сек)")

Сгенерировано 100 токенов за 4.90с (20.4 токенов/сек)


# 7. Визуализация: длина промпта vs количество токенов

Наглядно показать, что токен ≠ слово:

In [21]:
text = "Write an email apologizing to Sarah for the tragic gardening mishap."
words = text.split()
tokens = tokenizer.tokenize(text)
print(f"Слов: {len(words)}, токенов: {len(tokens)}")
print(tokens)

Слов: 11, токенов: 17
['▁Write', '▁an', '▁email', '▁apolog', 'izing', '▁to', '▁Sarah', '▁for', '▁the', '▁trag', 'ic', '▁garden', 'ing', '▁m', 'ish', 'ap', '.']


Для книги особенно ценны пункты 2 (визуализация логитов/вероятностей) и 3 (сравнение стратегий decoding) — это то, что визуальный стиль Alammar как раз любит показывать через диаграммы, и хорошо ложится в главу "Как работают трансформеры", которую ты сейчас пишешь: можно взять этот код как основу для практического примера, иллюстрирующего теорию.

# 8. Логиты для каждого сгенерированного токена (не только первого)

В пункте 2 мы смотрели вероятности только для первого токена после промпта. Теперь посмотрим, насколько модель была "уверена" на каждом шаге генерации:

In [24]:
outputs = model.generate(
    input_ids=input_ids,
    max_new_tokens=36,
    do_sample=False,
    output_scores=True,
    return_dict_in_generate=True,
)

for step, score in enumerate(outputs.scores):
    probs = score[0].softmax(dim=-1)
    top_prob, top_idx = probs.max(dim=-1)
    token = tokenizer.decode(top_idx)
    print(f"шаг {step:2d}: {token!r:15} уверенность {top_prob.item():.1%}")

шаг  0: 'Sub'           уверенность 85.9%
шаг  1: 'ject'          уверенность 100.0%
шаг  2: ':'             уверенность 100.0%
шаг  3: 'S'             уверенность 22.4%
шаг  4: 'inc'           уверенность 99.5%
шаг  5: 'ere'           уверенность 100.0%
шаг  6: 'Ap'            уверенность 99.8%
шаг  7: 'ologies'       уверенность 99.6%
шаг  8: 'for'           уверенность 93.4%
шаг  9: 'the'           уверенность 82.0%
шаг 10: 'Garden'        уверенность 79.5%
шаг 11: 'ing'           уверенность 56.6%
шаг 12: 'M'             уверенность 34.4%
шаг 13: 'ish'           уверенность 99.4%
шаг 14: 'ap'            уверенность 99.9%
шаг 15: '\n'            уверенность 92.5%
шаг 16: '\n'            уверенность 100.0%
шаг 17: '\n'            уверенность 95.0%
шаг 18: 'D'             уверенность 99.8%
шаг 19: 'ear'           уверенность 100.0%
шаг 20: 'Sarah'         уверенность 100.0%
шаг 21: ','             уверенность 100.0%
шаг 22: '\n'            уверенность 99.9%
шаг 23: '\n'            уве

Хорошо иллюстрирует, что модель не одинаково "уверена" на всех шагах — на очевидных продолжениях (например, после "Dear" почти наверняка идёт "Sarah") уверенность будет близка к 100%, а на развилках сюжета — заметно ниже.

# 9. Attention mask и что будет, если про него забыть

In [26]:
# Правильно — с attention_mask
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(output[0]))

# Что если передать только input_ids без attention_mask?
output_no_mask = model.generate(input_ids=inputs["input_ids"], max_new_tokens=20)
print(tokenizer.decode(output_no_mask[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear
Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear


Во втором случае Transformers обычно сам сгенерирует attention mask из единиц и выведет warning — хороший повод объяснить, зачем вообще нужна attention mask (в первую очередь актуально при батчинге с паддингом, где она отмечает, какие токены реальные, а какие — просто заполнитель).

# 10. Что будет, если скормить модели бессмысленный/случайный текст

In [27]:
garbage_prompt = "xkcd zzz 42 purple elephant quantum <|assistant|>"
input_ids_garbage = tokenizer(garbage_prompt, return_tensors="pt").input_ids.to("cuda")
output = model.generate(input_ids=input_ids_garbage, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(output[0]))

xkcd zzz 42 purple elephant quantum <|assistant|> In the realm of quantum mechanics, the concept of a purple elephant existing in a state of quantum superposition is a fascin


Показывает, что модель всё равно попытается сгенерировать что-то "правдоподобное" даже без осмысленного контекста — иллюстрация того, что LLM не "проверяет" осмысленность входа, а просто продолжает статистически вероятную последовательность.

# 11. System prompt: добавляем роль в messages

Вернёмся к формату messages (из более раннего примера с pipeline) и посмотрим, как system-сообщение меняет стиль ответа:

In [29]:
messages_variants = [
    [{"role": "user", "content": "Write an email apologizing to Sarah for the gardening mishap."}],
    [
        {"role": "system", "content": "You are a formal, corporate-style assistant."},
        {"role": "user", "content": "Write an email apologizing to Sarah for the gardening mishap."}
    ],
    [
        {"role": "system", "content": "You are a very casual, funny assistant."},
        {"role": "user", "content": "Write an email apologizing to Sarah for the gardening mishap."}
    ],
]

for msgs in messages_variants:
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(formatted, return_tensors="pt").input_ids.to("cuda")
    output = model.generate(input_ids=ids, max_new_tokens=80, do_sample=False)
    print(tokenizer.decode(output[0][ids.shape[1]:]))
    print("---")

Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have always admired the beauty and tranquility of your garden, and it
---
Subject: Sincere Apologies for the Gardening Incident


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have always admired the beauty and tranquility of your garden, and it was
---
Subject: Oopsie Daisy! My Gardening Blunder


Hey Sarah!


I hope this email finds you in high spirits and with a garden that's as lush as a rainforest. I've got to confess, I've been feeling a bit like a clumsy bumblebee in my own backyard.
---


Показывает apply_chat_template "вживую" (мы обсуждали этот метод в контексте pipeline раньше) и наглядно демонстрирует влияние system prompt на тон ответа.

# 12. Repetition penalty — борьба с зацикливанием

In [30]:
long_prompt = "List synonyms for the word 'happy'.<|assistant|>"
ids = tokenizer(long_prompt, return_tensors="pt").input_ids.to("cuda")

# Без штрафа за повторения
out1 = model.generate(input_ids=ids, max_new_tokens=80, do_sample=False)
print("Без repetition_penalty:", tokenizer.decode(out1[0][ids.shape[1]:]))

# С штрафом
out2 = model.generate(input_ids=ids, max_new_tokens=80, do_sample=False, repetition_penalty=1.3)
print("С repetition_penalty=1.3:", tokenizer.decode(out2[0][ids.shape[1]:]))

Без repetition_penalty: Here are some synonyms for the word 'happy':

1. Joyful
2. Delighted
3. Content
4. Pleased
5. Cheerful
6. Elated
7. Blissful
8. Ecstatic
9. Jubilant
10. Satisfied

These words all convey a sense
С repetition_penalty=1.3: Here are some common English words that can be used as substitutes or have similar meanings to "HAPPY":
1. Joyful 2: Contented3, Delighted4567890 Pleasant (when referring specifically about a situation) Cheerfully Blissfull Ecstatic Glad Pleased Jolly Merry Satisfied Thrilled Over


Greedy decoding нередко склонен зацикливаться на повторении одних и тех же фраз при длинной генерации — repetition_penalty штрафует уже использованные токены, снижая их вероятность на следующих шагах. Хорошая практическая иллюстрация проблемы, о которой редко рассказывают на пальцах.

# 13. Сколько памяти реально расходует каждый шаг генерации

Продолжение темы из твоих скриншотов про ресурсы Colab, но на уровне кода:

In [35]:
import torch

torch.cuda.reset_peak_memory_stats()
output = model.generate(input_ids=input_ids, max_new_tokens=3000, do_sample=False)
peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Пиковое потребление GPU-памяти во время генерации: {peak_mem:.2f} ГБ")

Пиковое потребление GPU-памяти во время генерации: 8.49 ГБ


Можно повторить с разным *max_new_tokens* (20, 100, 500) и построить график — покажет, что память растёт с длиной генерации (из-за роста KV-кэша), а не остаётся постоянной.

# 14. Свой собственный цикл генерации токен-за-токеном (реализация авторегрессии вручную)

Самое концептуально мощное упражнение — переписать model.generate() вручную, буквально повторяя то, что мы разбирали в самом начале ("выбранный токен добавляется к последовательности, и процесс повторяется"):

In [ ]:
ids = input_ids.clone()

for _ in range(2000):
    with torch.no_grad():
        outputs = model(ids)
    next_token_logits = outputs.logits[0, -1, :]
    next_token_id = next_token_logits.argmax().unsqueeze(0).unsqueeze(0)
    ids = torch.cat([ids, next_token_id], dim=1)
    print(tokenizer.decode(next_token_id[0]), end="")

Subject:HeartfeltApologiesfortheGardeningMishap


DearSarah,


Ihopethismessagefindsyouwell.Iamwritingtoexpressmydeepestapologiesfortheunfortunateincidentthatoccurredinyourgardenyesterday.


Asyouknow,Ihavealwaysadmiredthebeautyandtranquilityofyourgarden.ItwaswithgreatregretthatIdecidedtohelpyouwiththeweeding,hopingtocontributetothegarden'swell-being.However,inmyeagernesstoassist,Iaccidentallydamagedsomeofyourprizedroses.


ThemishaphappenedwhenIwasusingtheweedingtool,anditslippedfrommyhands,causingthedamage.Iunderstandthatthiswasnottheoutcomeyouhadhopedfor,andIamtrulysorryforanydistressordisappointmentthismayhavecausedyou.


PleaseknowthatIamwillingtotakefullresponsibilityformyactionsandamreadytomakeamendsinanywayyouseefit.Whetherit'sreplacingthedamagedplantsorhelpingyouwithanyadditionalgardeningtasks,Iamheretosupportyou.


Onceagain,Iapologizefortheunfortunateincidentandhopethatwecanmovepastthiswithunderstandingandforgiveness.


Warmregards,


[YourName]


---


Writeadetailedproposa

Это буквально "разворачивает" model.generate(do_sample=False) вручную, шаг за шагом — после этого упражнения читатель уже не будет воспринимать .generate() как чёрный ящик.



---



---



---



In [ ]:
print(input_ids)

tensor([[    1, 14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278,
         25305,   293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,
           920,   372,  9559, 29889, 32001]], device='cuda:0')


In [ ]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

<s>
Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [ ]:
generation_output

tensor([[    1, 14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278,
         25305,   293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,
           920,   372,  9559, 29889, 32001,  3323,   622, 29901,  1619,   317,
          3742,   406,  6225, 11763,   363,   278, 19906,   292,   341,   728,
           481,    13,    13, 29928,   799]], device='cuda:0')

In [ ]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparing Trained LLM Tokenizers


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [ ]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [ ]:
show_tokens(text, "bert-base-uncased")

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " / t / t " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

In [ ]:
show_tokens(text, "bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

In [ ]:
show_tokens(text, "gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

In [ ]:
show_tokens(text, "google/flan-t5-small")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600  </s> 

In [ ]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         "
 12 . 0 * 50 = 600 
 

In [ ]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]


 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [ ]:
show_tokens(text, "facebook/galactica-1.3b")

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [ ]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


<s>  
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [ ]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

In [ ]:
output.shape

torch.Size([1, 4, 384])

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


In [ ]:
output

tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)

# Text Embeddings (For Sentences and Whole Documents)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector.shape

(768,)

# Word Embeddings Beyond LLMs


In [ ]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [ ]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

# Recommending songs by embeddings

In [ ]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [ ]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [ ]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [ ]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2849', 0.9979680776596069),
 ('2640', 0.9964019060134888),
 ('3167', 0.9963980317115784),
 ('5549', 0.9959008693695068),
 ('2715', 0.9958351850509644),
 ('3117', 0.9954560995101929),
 ('2987', 0.9953479766845703),
 ('2881', 0.9951083660125732),
 ('2886', 0.9950577616691589),
 ('3094', 0.994985044002533)]

In [ ]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [ ]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
2849,Run To The Hills,Iron Maiden
2640,Red Barchetta,Rush
3167,Unchained,Van Halen
5549,November Rain,Guns N' Roses
2715,Rainbow In The Dark,Dio


In [ ]:
print_recommendations(2172)

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object
['2849' '2640' '3167' '5549' '2715']


,title,artist
id,,
2849,Run To The Hills,Iron Maiden
2640,Red Barchetta,Rush
3167,Unchained,Van Halen
5549,November Rain,Guns N' Roses
2715,Rainbow In The Dark,Dio


In [ ]:
print_recommendations(842)

title     California Love (w\/ Dr. Dre & Roger Troutman)
artist                                              2Pac
Name: 842 , dtype: object
['5668' '413' '5661' '330' '886']


,title,artist
id,,
5668,How We Do (w\/ 50 Cent),The Game
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
5661,Sweet Dreams,Beyonce
330,Hate It Or Love It (w\/ 50 Cent),The Game
886,Heartless,Kanye West
